In [16]:
pip install gymnasium_2048

In [17]:
import numpy as np
import torch
import random
import torch.nn as nn
import gymnasium as gym
import gymnasium_2048
import torch.optim as optim
import torch.nn.functional as F
from collections import deque


In [18]:
def parse_board(board):
    b = np.array(board, dtype=float).squeeze()
    if b.ndim == 3:
        axis = 0 if b.shape[0] > b.shape[-1] else -1
        b = np.argmax(b, axis=axis).astype(float)
        b = np.where(b > 0, 2 ** b, 0)
    elif b.ndim == 1 and b.size == 16:
        b = b.reshape((4, 4))
    return b


def state_tensor(board):
  b=parse_board(board).flatten()
  b_log=np.where(b>0 , np.log2(b),0)

  return torch.tensor(b_log,dtype=torch.float32).unsqueeze(0)


In [19]:
class DQN(nn.Module):
  def __init__(self,num_features,num_actions):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,num_actions)
    )
  def forward(self,features):
    return self.network(features)

class ReplayBuffer:
  def __init__(self,capacity=10000):
    self.buffer=deque(maxlen=capacity)

  def push(self, state,action,reward,next_state,done):
    self.buffer.append((state,action,reward,next_state,done))

  def sample(self,batch_size):
    batch=random.sample(self.buffer,batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    return torch.cat(states), torch.tensor(actions), torch.tensor(rewards, dtype=torch.float32), torch.cat(next_states), torch.tensor(dones, dtype=torch.float32)

  def __len__(self):
    return len(self.buffer)

In [20]:
class DQNAgent:
  def __init__(self):
    self.model=DQN(16, 4)
    self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
    self.memory=ReplayBuffer()

    self.gamma=0.99
    self.epsilon=1.0
    self.epsilon_min=0.05
    self.epsilon_decay=0.995
    self.batch_size=64

  def act(self,state,is_training=True):
    if is_training and random.random()<self.epsilon:
      return random.randint(0,3)

    with torch.no_grad():
      q_values=self.model(state)
      return torch.argmax(q_values).item()

  def train_step(self):
        if len(self.memory) < self.batch_size:
            return 0.0
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        q_values = self.model(states).gather(1, actions.unsqueeze(1)).squeeze(1)


        next_q_values = self.model(next_states).max(1)[0]

        targets = rewards + (self.gamma * next_q_values * (1 - dones))


        loss = F.mse_loss(q_values, targets)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return loss.item()

In [21]:
def train_unstable(episodes=500):
    env = gym.make('gymnasium_2048/TwentyFortyEight-v0')
    agent = DQNAgent()

    for episode in range(episodes):
        board, _ = env.reset()
        if isinstance(board, tuple): board = board[0]
        state = state_tensor(board)

        done = False
        total_reward = 0
        total_loss = 0
        steps = 0

        while not done:
            action = agent.act(state, is_training=True)
            next_board, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            if np.array_equal(parse_board(board), parse_board(next_board)):
                reward = -100.0

            next_state = state_tensor(next_board)

            agent.memory.push(state, action, reward, next_state, done)
            loss_val = agent.train_step()

            state = next_state
            board = next_board
            total_reward += reward
            total_loss += loss_val
            steps += 1

        agent.epsilon = max(agent.epsilon_min, agent.epsilon * agent.epsilon_decay)

        if (episode + 1) % 10 == 0:
            max_tile = int(np.max(parse_board(board)))
            avg_loss = total_loss / max(1, steps)

            empty_board_tensor = torch.zeros((1, 16))
            with torch.no_grad():
                q_guess = self_guess = agent.model(empty_board_tensor).mean().item()

            print(f"Ep {episode+1}/{episodes} | Max Tile: {max_tile} | Avg Loss: {avg_loss:.2f}")

    return agent

In [22]:

if __name__ == "__main__":
    train_unstable(episodes=200)

/tmp/ipykernel_8065/3865150327.py:14: RuntimeWarning: divide by zero encountered in log2
  b_log=np.where(b>0 , np.log2(b),0)


Ep 10/200 | Max Tile: 64 | Avg Loss: 1557.66
Ep 20/200 | Max Tile: 64 | Avg Loss: 1527.51
Ep 30/200 | Max Tile: 128 | Avg Loss: 1435.48
Ep 40/200 | Max Tile: 128 | Avg Loss: 1454.82
Ep 50/200 | Max Tile: 64 | Avg Loss: 1369.98
Ep 60/200 | Max Tile: 32 | Avg Loss: 1351.87
Ep 70/200 | Max Tile: 128 | Avg Loss: 1384.98
Ep 80/200 | Max Tile: 128 | Avg Loss: 1347.74
Ep 90/200 | Max Tile: 64 | Avg Loss: 1303.68
Ep 100/200 | Max Tile: 128 | Avg Loss: 1360.08
Ep 110/200 | Max Tile: 128 | Avg Loss: 1317.48
Ep 120/200 | Max Tile: 128 | Avg Loss: 1268.86
Ep 130/200 | Max Tile: 64 | Avg Loss: 1126.20
Ep 140/200 | Max Tile: 64 | Avg Loss: 1204.88
Ep 150/200 | Max Tile: 64 | Avg Loss: 1216.91
Ep 160/200 | Max Tile: 128 | Avg Loss: 1221.35
Ep 170/200 | Max Tile: 64 | Avg Loss: 1266.67
Ep 180/200 | Max Tile: 128 | Avg Loss: 1190.36
Ep 190/200 | Max Tile: 64 | Avg Loss: 1186.34
Ep 200/200 | Max Tile: 128 | Avg Loss: 1191.68
